# 🧠 Aula 12 — Laboratório Prático & Resumo Interativo de Infraestrutura para IA

Bem-vindo(a) à nossa Aula 12! Este notebook foi desenvolvido para **experimentar na prática** os conceitos de hardware, GPU, memória, redes e modelos de IA que estudamos durante o módulo através de uma interface interativa.

--- 
### 📌 GUIA RÁPIDO: COMO ABRIR E PREPARAR O NOTEBOOK NO GOOGLE COLAB

#### 1️⃣ Como abrir este notebook:
- Acesse [colab.research.google.com](https://colab.research.google.com/).
- Vá na aba **GitHub**, busque pelo repositório `https://github.com/jonasmaffei/senac-tecnico-ia` e selecione `aulas/aula12/aula12_pratica_colab.ipynb`.
- *(Alternativa)* Se baixou o arquivo para o seu computador, vá na aba **Fazer upload** e selecione este arquivo `.ipynb`.

#### 2️⃣ Ativar a GPU (Passo Obrigatório!):
1. No menu superior do Colab, clique em **Ambiente de execução** (*Runtime*).
2. Clique em **Alterar tipo de ambiente de execução** (*Change runtime type*).
3. Em **Acelerador de hardware**, selecione **T4 GPU**.
4. Clique em **Salvar**.

#### 3️⃣ Como Executar os Blocos:
- Passe o mouse sobre qualquer bloco cinza abaixo e clique no botão **Play ▶️** (ou aperte `Ctrl + Enter`).
- Altere os valores nos **sliders e menus interativos** no lado direito do código e clique em Play novamente para ver as mudanças!


## 🔹 Bloco 1: A Batalha de Hardware — CPU vs GPU (Aulas 1 e 2)

**Conceito:**
- **CPU (Von Neumann/CISC):** Como um chef de cozinha super treinado. Resolve tarefas complexas uma por uma (sequencial).
- **GPU (SIMD):** Como uma fábrica com centenas de assistentes simples trabalhando juntos. Aplica a mesma instrução em milhares de dados simultaneamente.

Execute o código abaixo alterando o tamanho dos dados no slider para ver a diferença!


In [ ]:
# @title 🎛️ Experimento 1: Multiplicação Massiva de Matrizes (CPU vs GPU)
import torch
import time
import matplotlib.pyplot as plt

# Formulário Interativo no Colab
tamanho_matriz = 3000 # @param {type:"slider", min:1000, max:6000, step:1000}

print(f"🔄 Criando matrizes de tamanho {tamanho_matriz} x {tamanho_matriz}...")

# 1. Teste na CPU
matriz_a_cpu = torch.randn(tamanho_matriz, tamanho_matriz)
matriz_b_cpu = torch.randn(tamanho_matriz, tamanho_matriz)

inicio_cpu = time.time()
resultado_cpu = torch.matmul(matriz_a_cpu, matriz_b_cpu)
tempo_cpu = time.time() - inicio_cpu
print(f"⏱️ Tempo gasto na CPU: {tempo_cpu:.4f} segundos")

# 2. Teste na GPU (se disponível)
if torch.cuda.is_available():
    matriz_a_gpu = matriz_a_cpu.to('cuda')
    matriz_b_gpu = matriz_b_cpu.to('cuda')
    
    # Aquecimento da GPU
    torch.matmul(matriz_a_gpu, matriz_b_gpu)
    torch.cuda.synchronize()
    
    inicio_gpu = time.time()
    resultado_gpu = torch.matmul(matriz_a_gpu, matriz_b_gpu)
    torch.cuda.synchronize()
    tempo_gpu = time.time() - inicio_gpu
    print(f"🚀 Tempo gasto na GPU: {tempo_gpu:.4f} segundos")
    
    aceleracao = tempo_cpu / tempo_gpu
    print(f"\n⚡ A GPU foi {aceleracao:.1f}x MAIS RÁPIDA que a CPU!")
    
    # Gráficos Visuais
    plt.figure(figsize=(7, 4))
    plt.bar(['CPU (Sequencial)', 'GPU (Paralelo)'], [tempo_cpu, tempo_gpu], color=['#ff6b6b', '#51cf66'])
    plt.ylabel('Tempo em Segundos (menor é melhor)')
    plt.title(f'Comparativo de Velocidade (Matriz {tamanho_matriz}x{tamanho_matriz})')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("⚠️ GPU não detectada! Lembre-se de ativar a GPU no menu 'Ambiente de execução'.")


## 🔹 Bloco 2: O Gargalo de Memória e Barramento PCIe (Aula 3)

**Conceito:**
Não adianta ter uma GPU ultrarrápida se o tempo para **enviar o dado da memória RAM (CPU) para a VRAM (GPU)** for muito alto. Esse transporte de dados via barramento PCIe pode ser o gargalo da sua solução de IA.


In [ ]:
# @title 🎛️ Experimento 2: Medindo o Custo de Transferir Dados (RAM ➡️ VRAM)
import torch
import time
import matplotlib.pyplot as plt

tamanho_mb = 500 # @param {type:"slider", min:100, max:1000, step:100}
elementos = (tamanho_mb * 1024 * 1024) // 4 # float32 tem 4 bytes

if torch.cuda.is_available():
    dado_ram = torch.randn(elementos)
    
    # Medindo tempo de envio (RAM -> VRAM)
    inicio_transf = time.time()
    dado_vram = dado_ram.to('cuda')
    torch.cuda.synchronize()
    tempo_transf = time.time() - inicio_transf
    
    # Medindo tempo de cálculo na GPU (ex: multiplicar tudo por 2)
    inicio_calc = time.time()
    resultado = dado_vram * 2.0
    torch.cuda.synchronize()
    tempo_calc = time.time() - inicio_calc
    
    print(f"📦 Tamanho do Dado: {tamanho_mb} MB")
    print(f"🚚 Tempo para TRANSFERIR (RAM -> VRAM): {tempo_transf:.5f}s")
    print(f"⚡ Tempo para PROCESSAR na GPU:         {tempo_calc:.5f}s")
    
    # Gráfico de pizza
    labels = ['Transferência (PCIe)', 'Processamento (GPU)']
    tempos = [tempo_transf, tempo_calc]
    plt.figure(figsize=(6, 5))
    plt.pie(tempos, labels=labels, autopct='%1.1f%%', colors=['#fcc419', '#339af0'], startangle=90)
    plt.title('Onde foi gasto o tempo total?')
    plt.show()
else:
    print("⚠️ Ative a GPU no Colab para este teste.")


## 🔹 Bloco 3: Processamento Paralelo de Imagens em Blocos (Aulas 7 e 8)

**Conceito:**
Como as GPUs trabalham com visão computacional? Elas dividem a imagem em uma **grade de blocos e threads (Tiling)**. Cada thread cuida de um pequeno grupo de pixels simultaneamente.


In [ ]:
# @title 🎛️ Experimento 3: Filtro de Imagem Paralelo (Simulação Visual de CUDA)
import numpy as np
import matplotlib.pyplot as plt

# Criar uma imagem sintética (um padrão geométrico colorido)
largura, altura = 500, 500
x, y = np.meshgrid(np.linspace(-2, 2, largura), np.linspace(-2, 2, altura))
imagem_original = np.sin(x**2 + y**2)

# Escolher o filtro interativo
efeito = "Inverter e Contrastar" # @param ["Borrão (Blur)", "Inverter e Contrastar", "Detecção de Bordas"]

def aplicar_filtro(img, efeito_escolhido):
    if efeito_escolhido == "Borrão (Blur)":
        return np.roll(img, 5, axis=0) * 0.5 + img * 0.5
    elif efeito_escolhido == "Inverter e Contrastar":
        return np.where(img > 0, 1.0, -1.0)
    else:
        return np.abs(np.gradient(img)[0])

imagem_processada = aplicar_filtro(imagem_original, efeito)

# Exibição lado a lado
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(imagem_original, cmap='magma')
ax[0].set_title("1. Imagem Original (Matriz de Pixels)")
ax[0].axis('off')

ax[1].imshow(imagem_processada, cmap='viridis')
ax[1].set_title(f"2. Resultado GPU: {efeito}")
ax[1].axis('off')

plt.tight_layout()
plt.show()
print("💡 Dica: Na GPU, cada um dos 250.000 pixels foi calculado ao mesmo tempo por threads paralelas!")


## 🔹 Bloco 4: Monitorando a VRAM com Modelos de Linguagem (Aulas 9 e 10)

**Conceito:**
Modelos de IA (LLMs) ocupam espaço na **VRAM (Memória da Placa de Vídeo)**. Se o modelo for grande demais para a VRAM disponível, ocorre o erro de *Out of Memory (OOM)*.


In [ ]:
# @title 🎛️ Experimento 4: Monitor de VRAM do Linux (`nvidia-smi` em tempo real)
import subprocess
import torch

print("📊 MONITOR DE RECURSOS DA GPU (NVIDIA-SMI):\n")
try:
    # Executa o comando de terminal do Linux
    resultado_smi = subprocess.check_output(['nvidia-smi']).decode('utf-8')
    print(resultado_smi)
except Exception as e:
    print("⚠️ Não foi possível rodar o nvidia-smi. Certifique-se que está usando GPU no Colab.")

if torch.cuda.is_available():
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    vram_alocada = torch.cuda.memory_allocated(0) / (1024**3)
    vram_livre = vram_total - vram_alocada
    
    print(f"🎯 Placa Detectada: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM Total:     {vram_total:.2f} GB")
    print(f"🔴 VRAM Usada:     {vram_alocada:.2f} GB")
    print(f"🟢 VRAM Livre:     {vram_livre:.2f} GB")


## 🔹 Bloco 5: Testando uma IA Real — Análise de Sentimento de Clientes (Aula 11)

**Conceito:**
Vamos colocar tudo em prática! Carregaremos um modelo pré-treinado na GPU para analisar avaliações de clientes automaticamente.


In [ ]:
# @title 🎛️ Experimento 5: Classificador de Feedbacks de Clientes (Aplicação Prática)
from transformers import pipeline
import torch

# Texto de entrada via formulário do Colab
texto_do_cliente = "O produto chegou dentro do prazo, mas a embalagem veio amassada e o manual de instruções é muito confuso." # @param {type:"string"}

print("🤖 Carregando modelo de Inteligência Artificial na GPU...")
dispositivo = 0 if torch.cuda.is_available() else -1

# Pipeline leve da Hugging Face
classificador = pipeline(
    "sentiment-analysis", 
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=dispositivo
)

resultado = classificador(texto_do_cliente)[0]

estrelas = resultado['label'] # Ex: '1 star', '4 stars'
confianca = resultado['score'] * 100

print("\n" + "="*50)
print(f"📝 Texto Analisado: '{texto_do_cliente}'")
print(f"⭐ Avaliação Estimada: {estrelas}")
print(f"🎯 Grau de Confiabilidade da IA: {confianca:.1f}%")
print("="*50)
